# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [4]:
# The .env file lives in this folder (project/starter) and holds OPENAI_API_KEY,
# OPENAI_BASE_URL and TAVILY_API_KEY. The setup cell above already loaded and checked it.
print("Environment OK:", os.getenv("OPENAI_BASE_URL"))

Environment OK: https://openai.vocareum.com/v1


### VectorDB Instance

In [5]:
# Persistent ChromaDB client. Part 2 loads the database from the same "chromadb" path.
chroma_client = chromadb.PersistentClient(path="chromadb")
chroma_client

### Collection

In [6]:
# Part 2 rebuilds this same function when it loads the collection.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [7]:
# Collection name "udaplay" is what Part 2 loads.
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)
print("Collection:", collection.name, "| documents so far:", collection.count())

Collection: udaplay | documents so far: 0


### Add documents

In [8]:
# Run this notebook from project/starter: the game files are in ./games
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Text that gets embedded: platform, name, year, genre, publisher and description,
    # so questions about any of these fields can match semantically.
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
        f"{game['Genre']} game published by {game['Publisher']}. {game['Description']}"
    )

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    # upsert (not add) so re-running the notebook doesn't fail on duplicate ids
    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

print("Documents in collection:", collection.count())

Documents in collection: 15


### Semantic search
Query the collection with natural-language questions. Lower distance means a closer match.

In [9]:
def show(query, n_results=3):
    res = collection.query(query_texts=[query], n_results=n_results)
    print(f"Q: {query}")
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"  dist={dist:.3f} | {meta['Name']} ({meta['Platform']}, {meta['YearOfRelease']}) - {meta['Publisher']}")
    print()

show("Who published Grand Theft Auto: San Andreas?")
show("When was Pokémon Gold and Silver released?")
show("Which was the first 3D platformer Mario game?")
show("racing simulator on PlayStation")
show("Was Mortal Kombat X released for PlayStation 5?")  # not in the dataset

Q: Who published Grand Theft Auto: San Andreas?
  dist=0.118 | Grand Theft Auto: San Andreas (PlayStation 2, 2004) - Rockstar Games
  dist=0.201 | Gran Turismo (PlayStation 1, 1997) - Sony Computer Entertainment
  dist=0.204 | Gran Turismo 5 (PlayStation 3, 2010) - Sony Computer Entertainment



Q: When was Pokémon Gold and Silver released?
  dist=0.111 | Pokémon Gold and Silver (Game Boy Color, 1999) - Nintendo
  dist=0.148 | Pokémon Ruby and Sapphire (Game Boy Advance, 2002) - Nintendo
  dist=0.226 | Super Mario 64 (Nintendo 64, 1996) - Nintendo



Q: Which was the first 3D platformer Mario game?
  dist=0.109 | Super Mario 64 (Nintendo 64, 1996) - Nintendo
  dist=0.134 | Super Mario World (Super Nintendo Entertainment System (SNES), 1990) - Nintendo
  dist=0.197 | Mario Kart 8 Deluxe (Nintendo Switch, 2017) - Nintendo



Q: racing simulator on PlayStation
  dist=0.117 | Gran Turismo (PlayStation 1, 1997) - Sony Computer Entertainment
  dist=0.117 | Gran Turismo 5 (PlayStation 3, 2010) - Sony Computer Entertainment
  dist=0.187 | Mario Kart 8 Deluxe (Nintendo Switch, 2017) - Nintendo



Q: Was Mortal Kombat X released for PlayStation 5?
  dist=0.209 | Marvel's Spider-Man 2 (PlayStation 5, 2023) - Sony Interactive Entertainment
  dist=0.224 | Gran Turismo 5 (PlayStation 3, 2010) - Sony Computer Entertainment
  dist=0.224 | Marvel's Spider-Man (PlayStation 4, 2018) - Sony Interactive Entertainment



### Reusable vector store manager
The same logic is packaged in `lib/game_store.py` as `GameVectorStore`: it loads the JSON files, indexes them, and exposes `search()`. Part 2 uses it so both notebooks share one definition of the database.

In [10]:
from lib.game_store import GameVectorStore

store = GameVectorStore()          # same path ("chromadb") and collection ("udaplay")
print("Indexed:", store.index_games("games"), "games | total in collection:", store.count())

for hit in store.search("What platform was Super Mario 64 launched on?", n_results=2):
    print(f"dist={hit['distance']:.3f} | {hit['document']}")

Indexed: 15 games | total in collection: 15


dist=0.099 | [Nintendo 64] Super Mario 64 (1996) - Platformer game published by Nintendo. A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
dist=0.145 | [Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - Platformer game published by Nintendo. A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.
